# 多层感知机：$y=x^2$ 的计算过程

这一篇用一个样本手工走完整个计算过程：前向传播、反向传播，以及一次梯度下降更新。


## 单个样本的手推版本

先只看一个样本：

$$
x=2, \quad y=4
$$

数学推导里默认把向量写成列向量。这个例子里输入维度 $D=1$，隐藏层宽度 $I=2$，输出维度 $J=1$，所以各变量尺寸是：

| 变量 | 含义 | 数学尺寸 | 本例尺寸 |
|---|---|---:|---:|
| $x$ | 单个样本输入 | $D\times 1$ | $1\times 1$ |
| $W_1$ | 输入层到隐藏层权重 | $I\times D$ | $2\times 1$ |
| $b_1$ | 隐藏层偏置 | $I\times 1$ | $2\times 1$ |
| $z_1$ | 隐藏层线性输出 | $I\times 1$ | $2\times 1$ |
| $h$ | 隐藏层激活 | $I\times 1$ | $2\times 1$ |
| $W_2$ | 隐藏层到输出层权重 | $J\times I$ | $1\times 2$ |
| $b_2$ | 输出层偏置 | $J\times 1$ | $1\times 1$ |
| $\hat y$ | 模型输出 | $J\times 1$ | $1\times 1$ |
| $y$ | 真实标签 | $J\times 1$ | $1\times 1$ |

初始化参数：

$$
W_1=\begin{bmatrix}1\\-1\end{bmatrix}, \quad
b_1=\begin{bmatrix}0\\0\end{bmatrix}, \quad
W_2=\begin{bmatrix}1&1\end{bmatrix}, \quad
b_2=0
$$

前向传播：

$$
z_1=W_1x+b_1
$$

$$
h=\operatorname{ReLU}(z_1)
$$

$$
\hat y=W_2h+b_2
$$

$$
L=\frac{1}{2}(\hat y-y)^2
$$

下面代码为了贴近 NumPy / PyTorch 等深度学习框架的 batch 写法，实际把单个样本写成行向量；它和上面的列向量公式互为转置。

这是一个很常见的标准做法：数学推导和线性代数教材通常默认向量是列向量，方便写成 $Wx+b$；工程代码通常把 batch 维度放在第 0 维，让输入形状成为 `(batch_size, input_dim)`，所以每个样本自然就是矩阵中的一行。两种写法表达的是同一个计算，只是组织数据的方向不同。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

font_path = Path("/System/Library/Fonts/Hiragino Sans GB.ttc")
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams["font.family"] = "Hiragino Sans GB"
plt.rcParams["axes.unicode_minus"] = False


def relu(x):
    return np.maximum(0, x)


def relu_grad(x):
    return (x > 0).astype(float)


x = np.array([[2.0]])       # shape: (1, 1)
y = np.array([[4.0]])       # shape: (1, 1)

W1 = np.array([[1.0, -1.0]])
b1 = np.array([[0.0, 0.0]])
W2 = np.array([[1.0], [1.0]])
b2 = np.array([[0.0]])

z1 = np.dot(x, W1) + b1
h = relu(z1)
y_hat = np.dot(h, W2) + b2
loss = 0.5 * (y_hat - y) ** 2

print("z1    =", z1)
print("h     =", h)
print("y_hat =", y_hat)
print("loss  =", loss)


z1    = [[ 2. -2.]]
h     = [[2. 0.]]
y_hat = [[2.]]
loss  = [[2.]]


## 反向传播计算

从损失函数开始往回算梯度。代码采用 batch 行向量写法，所以输出层权重梯度对应 `h.T @ dy`，隐藏层权重梯度对应 `x.T @ dz1`。


In [2]:
dy = y_hat - y

dW2 = np.dot(h.T, dy)
db2 = dy.sum(axis=0, keepdims=True)

dh = np.dot(dy, W2.T)
dz1 = dh * relu_grad(z1)

dW1 = np.dot(x.T, dz1)
db1 = dz1.sum(axis=0, keepdims=True)

print("dy  =", dy)
print("dW2 =\n", dW2)
print("db2 =", db2)
print("dh  =", dh)
print("dz1 =", dz1)
print("dW1 =", dW1)
print("db1 =", db1)


dy  = [[-2.]]
dW2 =
 [[-4.]
 [ 0.]]
db2 = [[-2.]]
dh  = [[-2. -2.]]
dz1 = [[-2. -0.]]
dW1 = [[-4.  0.]]
db1 = [[-2.  0.]]


这个例子里第二个隐藏神经元的 $z_1=-2$，经过 ReLU 后输出为 0，所以它对应的梯度也被截断为 0。

## 参数更新

得到梯度以后，用最普通的梯度下降更新参数：

$$
\theta_{next}=\theta-\eta\frac{\partial L}{\partial \theta}
$$

其中 $\eta$ 是学习率。


In [3]:
lr = 0.1

W1_next = W1 - lr * dW1
b1_next = b1 - lr * db1
W2_next = W2 - lr * dW2
b2_next = b2 - lr * db2

print("W1_next =", W1_next)
print("b1_next =", b1_next)
print("W2_next =\n", W2_next)
print("b2_next =", b2_next)


W1_next = [[ 1.4 -1. ]]
b1_next = [[0.2 0. ]]
W2_next =
 [[1.4]
 [1. ]]
b2_next = [[0.2]]
